# Business Entity Resolution: v2 (two-stage, per-business decisions)

**How to run:** click **Save Version** (top right), choose **Save & Run All (Commit)**, then **Save**. Settings: CPU (no accelerator), Internet On. It runs on Kaggle's servers even if you close the tab.

What is new compared with v1:
1. **Stage 1** is v1's model, trained on 200k businesses (v1: 150k). Each candidate gets a chance of being a true match.
2. **Stage 2** looks at all candidates of one business together, from Source 2 and Source 3. A business usually appears about 3 times and those copies look alike, so a candidate that resembles the other strong candidates gets a boost; an odd one out gets pushed down.
3. **Per-business decision:** keep a candidate only if its chance is high enough **and** close enough to that business's best candidate (both limits are tuned on held-out businesses).

Cells: 1 install, 2 v1 pipeline code, 3 settings, 4 v2 code, 5 **train** (prints `STAGE-2 BEST val F0.5`; v1 was 0.9529), 6 **predict** test and write `output/matching_results.tsv`, 7 official format check.

Expect roughly 1.5 hours for training and 2 to 3 hours for prediction. No external data or APIs are used, only the competition files.

In [ ]:
!pip install -q rapidfuzz sparse_dot_topn unidecode polars lightgbm

In [ ]:
# =====================================================================
# Business Entity Resolution: full pipeline (clean -> shortlist -> features -> LightGBM -> decide)
# No external data or services are used anywhere. Only the provided files.
# =====================================================================
import os, re, gc, glob, time, json
import numpy as np
import pandas as pd
import polars as pl
from multiprocessing import Pool
from unidecode import unidecode
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from rapidfuzz.process import cpdist
import lightgbm as lgb

N_CPU = os.cpu_count() or 4

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

# ---------------------------------------------------------------------
# 1. Loading. Files are tab-separated, all columns read as text.
#    Handles files split into *_part_N.tsv (BOM / CRLF / repeated header).
# ---------------------------------------------------------------------
def find_file(root, name):
    hits = sorted(glob.glob(f"{root}/**/{name}.tsv", recursive=True))
    if hits:
        return [hits[0]]
    parts = glob.glob(f"{root}/**/{name}_part_*.tsv", recursive=True)
    parts.sort(key=lambda p: int(re.findall(r"_part_(\d+)", p)[-1]))
    if not parts:
        raise FileNotFoundError(f"{name}.tsv not found under {root}")
    return parts

def load(root, name):
    frames, header = [], None
    for p in find_file(root, name):
        txt = open(p, encoding="utf-8-sig").read().replace("\r\n", "\n").replace("\r", "")
        if txt.startswith(("entity_id\t", "source1_entity_id\t")):
            header, txt = txt.split("\n", 1)
        frames.append(txt)
    body = "".join(f if f.endswith("\n") else f + "\n" for f in frames)
    df = pl.read_csv((header + "\n" + body).encode(), separator="\t", quote_char=None, infer_schema_length=0)
    return df.fill_null("").to_pandas()

# ---------------------------------------------------------------------
# 2. Cleaning (normalisation). Hand-written rules only.
# ---------------------------------------------------------------------
ABBR = {"rd":"road","st":"street","ave":"avenue","av":"avenue","blvd":"boulevard","bd":"boulevard","hwy":"highway",
        "dr":"drive","ln":"lane","ct":"court","pl":"place","pkwy":"parkway","cir":"circle","ste":"suite","fl":"floor",
        "apt":"apartment","no":"number","nr":"near","opp":"opposite","bldg":"building","ind":"industrial",
        "dist":"district","r":"rue","ch":"chemin","n":"north","s":"south","e":"east","w":"west","mt":"mount",
        "ft":"fort","hts":"heights","sq":"square","imp":"impasse","all":"allee"}
NULLS = {"null", "na", "none", "nan"}
# legal words, written in "skeleton" form (see skel_tok): covers Limited/Ltd/limittedd, Private/Pvt/praaivett, LLC, SARL ...
LEGAL_SK = {"lmtd","ltd","prwt","pwt","llc","llp","inc","ncrprtd","crp","crprtn","c","cmpny","plc","srl","ss","sa",
            "sc","erl","snc","gmbh","t","m","s","d","dba","lp","pc","pllc","pa","ms","and"}

def base(s):
    s = unidecode(str(s)).lower()                                   # also turns Hindi/Telugu/... script into Latin letters
    s = re.sub(r"<?null>?|n/a", " ", s)
    s = re.sub(r"https?://|www\.|\.(com|net|org|in|co|io|biz|fr|us)\b", " ", s)
    s = s.replace("&", " and ").replace("+", " and ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def skel_tok(t):
    """Spelling-robust 'skeleton' of a word: drop vowels, merge ph/f, bh/b ..., collapse double letters.
    'limited' and the transliterated 'limittedd' both become 'lmtd'."""
    if not t or t.isdigit():
        return t
    t = t.replace("0", "o").replace("1", "l") if re.search(r"[a-z]", t) else t   # leetspeak: C1ean -> Clean
    t = re.sub(r"ph", "f", t)
    t = re.sub(r"([bkgtdcs])h", r"\1", t)
    t = t.replace("v", "w").replace("q", "k").replace("z", "s").replace("x", "ks")
    t = t[0] + re.sub(r"[aeiouy]", "", t[1:])
    return re.sub(r"(.)\1+", r"\1", t)

US = {"alabama":"al","alaska":"ak","arizona":"az","arkansas":"ar","california":"ca","colorado":"co","connecticut":"ct",
"delaware":"de","florida":"fl","georgia":"ga","hawaii":"hi","idaho":"id","illinois":"il","indiana":"in","iowa":"ia",
"kansas":"ks","kentucky":"ky","louisiana":"la","maine":"me","maryland":"md","massachusetts":"ma","michigan":"mi",
"minnesota":"mn","mississippi":"ms","missouri":"mo","montana":"mt","nebraska":"ne","nevada":"nv","new hampshire":"nh",
"new jersey":"nj","new mexico":"nm","new york":"ny","north carolina":"nc","north dakota":"nd","ohio":"oh","oklahoma":"ok",
"oregon":"or","pennsylvania":"pa","rhode island":"ri","south carolina":"sc","south dakota":"sd","tennessee":"tn",
"texas":"tx","utah":"ut","vermont":"vt","virginia":"va","washington":"wa","west virginia":"wv","wisconsin":"wi",
"wyoming":"wy","district of columbia":"dc","puerto rico":"pr"}
IN = {"andhra pradesh":"ap","arunachal pradesh":"ar","assam":"as","bihar":"br","chhattisgarh":"cg","goa":"ga",
"gujarat":"gj","haryana":"hr","himachal pradesh":"hp","jharkhand":"jh","karnataka":"ka","kerala":"kl",
"madhya pradesh":"mp","maharashtra":"mh","manipur":"mn","meghalaya":"ml","mizoram":"mz","nagaland":"nl","odisha":"od",
"orissa":"od","punjab":"pb","rajasthan":"rj","sikkim":"sk","tamil nadu":"tn","telangana":"ts","tripura":"tr",
"uttar pradesh":"up","uttarakhand":"uk","uttaranchal":"uk","west bengal":"wb","delhi":"dl","new delhi":"dl",
"jammu and kashmir":"jk","ladakh":"la","puducherry":"py","pondicherry":"py","chandigarh":"ch",
"andaman and nicobar islands":"an","lakshadweep":"ld"}
MERGE_ST = {"ts": "ap"}            # Telangana was split from Andhra Pradesh; the data mixes them
_STATE = {}
for _d in (US, IN):
    for _full, _code in _d.items():
        _STATE[tuple(skel_tok(t) for t in _full.split())] = _code
_CODES = set(US.values()) | set(IN.values())

def states_of(raw_addr):
    """All state/province codes mentioned in an address ('' when none). Country-agnostic: unknown countries get ''."""
    found = set()
    for seg in unidecode(str(raw_addr)).lower().split(","):
        toks = re.sub(r"[^a-z ]", " ", seg).split()
        if not toks:
            continue
        if len(toks) == 1 and toks[0] in _CODES:
            found.add(toks[0]); continue
        sk = [skel_tok(t) for t in toks]
        for n in (4, 3, 2, 1):
            for i in range(len(sk) - n + 1):
                key = tuple(sk[i:i + n])
                if key in _STATE and (n > 1 or len(key[0]) >= 3):
                    found.add(_STATE[key])
    return "|".join(sorted({MERGE_ST.get(x, x) for x in found}))

def _clean_rows(rows):
    out = []
    for name, addr in rows:
        nb = base(name)
        sk = [skel_tok(t) for t in nb.split()]
        core = [t for t in sk if t not in LEGAL_SK] or sk
        an = " ".join(ABBR.get(t, t) for t in base(addr).split() if t not in NULLS)
        out.append((nb, " ".join(core), "".join(core), an,
                    " ".join(skel_tok(t) for t in an.split()), states_of(addr)))
    return out

def prepare(df):
    """Adds the cleaned columns. Runs on all CPU cores."""
    rows = list(zip(df.business_name.tolist(), df.business_address.tolist()))
    step = 50_000
    with Pool(N_CPU) as pool:
        res = pool.map(_clean_rows, [rows[i:i + step] for i in range(0, len(rows), step)])
    res = [r for chunk in res for r in chunk]
    cols = ["name_b", "name_sk", "name_skc", "addr_n", "addr_sk", "states"]
    for j, c in enumerate(cols):
        df[c] = [r[j] for r in res]
    df.drop(columns=["business_address"], inplace=True)
    df["nonascii"] = df.business_name.str.contains(r"[^\x00-\x7f]", regex=True).astype(np.int8)
    df.drop(columns=["business_name"], inplace=True)
    return df.reset_index(drop=True)

# ---------------------------------------------------------------------
# 3. Shortlist ("blocking"). For each Source 1 record, look only inside the same country and
#    the same state, and keep the top-k most similar names and the top-k most similar addresses.
#    Records whose state could not be read form a separate pool per country.
# ---------------------------------------------------------------------
def _vec_name():
    return TfidfVectorizer(analyzer="char", ngram_range=(3, 3), dtype=np.float32, sublinear_tf=True)
def _vec_addr():
    return TfidfVectorizer(analyzer="word", token_pattern=r"\S+", ngram_range=(1, 2), dtype=np.float32, sublinear_tf=True)

class Index:
    """TF-IDF indexes over one pool of candidate records (one country of Source 2 or Source 3)."""
    def __init__(self, B, k_state=15, k_unk=10):
        self.B, self.k_state, self.k_unk = B, k_state, k_unk
        self.cache = {}
        st = B.states.str.split("|")
        self.members = {}
        for pos, lst in zip(range(len(B)), st):
            for s in lst:
                if s:
                    self.members.setdefault(s, []).append(pos)
        self.unknown = np.flatnonzero(B.states.to_numpy() == "")

    def _build(self, key, pos):
        if key in self.cache:
            return self.cache[key]
        pos = np.asarray(pos, dtype=np.int64)
        entry = {"pos": pos}
        for col, mk in (("name_skc", _vec_name), ("addr_sk", _vec_addr)):
            try:
                v = mk(); M = v.fit_transform(self.B[col].to_numpy()[pos])
                entry[col] = (v, M.T.tocsr())
            except ValueError:            # empty vocabulary, e.g. every address blank
                entry[col] = None
        self.cache[key] = entry
        return entry

    def query(self, A, thr=0.1, chunk=40_000):
        """Returns DataFrame(a, b, sim_name, sim_addr); a = row in A, b = row in self.B."""
        out = []
        ast = A.states.str.split("|").tolist()
        groups = {}
        for i, lst in enumerate(ast):
            for s in lst:
                groups.setdefault(s, []).append(i)
        jobs = []
        for s, rows in groups.items():
            if s == "":
                jobs.append(("__all__", np.arange(len(self.B)), rows, self.k_state))
            elif s in self.members:
                jobs.append((s, self.members[s], rows, self.k_state))
        known = [i for i, lst in enumerate(ast) if lst != [""]]
        if len(self.unknown) and known:
            jobs.append(("__unknown__", self.unknown, known, self.k_unk))
        for key, pos, rows, k in jobs:
            e = self._build(key, pos)
            rows = np.asarray(rows)
            for col, tag in (("name_skc", 0), ("addr_sk", 1)):
                if e[col] is None:
                    continue
                v, BT = e[col]
                texts = A[col].to_numpy()[rows]
                for i in range(0, len(rows), chunk):
                    C = sp_matmul_topn(v.transform(texts[i:i + chunk]), BT, top_n=k, threshold=thr,
                                       n_threads=N_CPU).tocoo()
                    out.append((rows[i + C.row], e["pos"][C.col], C.data, tag))
        if not out:
            return pd.DataFrame({"a": [], "b": [], "sim_name": [], "sim_addr": []})
        a = np.concatenate([o[0] for o in out]).astype(np.int64)
        b = np.concatenate([o[1] for o in out]).astype(np.int64)
        sim = np.concatenate([o[2] for o in out]).astype(np.float32)
        tag = np.concatenate([np.full(len(o[0]), o[3], np.int8) for o in out])
        df = pd.DataFrame({"key": a * len(self.B) + b,
                           "sim_name": np.where(tag == 0, sim, np.nan),
                           "sim_addr": np.where(tag == 1, sim, np.nan)})
        df = df.groupby("key", sort=False).max().reset_index()
        df["a"] = (df.key // len(self.B)).astype(np.int32)
        df["b"] = (df.key % len(self.B)).astype(np.int32)
        return df.drop(columns="key")

# ---------------------------------------------------------------------
# 4. Features: how similar is each (Source 1, candidate) pair?
# ---------------------------------------------------------------------
FULL = [("tset", fuzz.token_set_ratio), ("tsort", fuzz.token_sort_ratio), ("ratio", fuzz.ratio),
        ("partial", fuzz.partial_ratio), ("jw", JaroWinkler.normalized_similarity)]

def _nums(s):
    return set(re.findall(r"\d+", s))

def features(P, A, B, is_s3):
    a, b = P.a.to_numpy(), P.b.to_numpy()
    g = lambda D, col, idx: D[col].to_numpy()[idx]
    f = pd.DataFrame({"sim_name": P.sim_name.to_numpy(), "sim_addr": P.sim_addr.to_numpy()})
    for col, pre, scorers in (("name_b", "nb", FULL), ("name_sk", "nsk", FULL),
                              ("name_skc", "nskc", [("ratio", fuzz.ratio), ("partial", fuzz.partial_ratio)]),
                              ("addr_sk", "ask", [("tset", fuzz.token_set_ratio), ("ratio", fuzz.ratio),
                                                  ("partial", fuzz.partial_ratio)])):
        x, y = g(A, col, a).tolist(), g(B, col, b).tolist()
        for nm, sc in scorers:
            f[f"{pre}_{nm}"] = cpdist(x, y, scorer=sc, workers=N_CPU).astype(np.float32)
    xa, ya = g(A, "addr_n", a), g(B, "addr_n", b)
    nl = [_nums(s) for s in xa]; nr = [_nums(s) for s in ya]
    f["num_jacc"] = np.array([len(p & q) / len(p | q) if (p | q) else -1 for p, q in zip(nl, nr)], np.float32)
    f["num_contain"] = np.array([len(p & q) / len(q) if q else -1 for p, q in zip(nl, nr)], np.float32)
    sa, sb = g(A, "states", a), g(B, "states", b)
    f["state_rel"] = np.array([-1 if (not p or not q) else (1 if set(p.split("|")) & set(q.split("|")) else 0)
                               for p, q in zip(sa, sb)], np.int8)
    f["name_exact"] = (g(A, "name_skc", a) == g(B, "name_skc", b)).astype(np.int8)
    f["cand_addr_len"] = np.fromiter((len(s) for s in ya), np.int32, len(ya))
    f["s1_addr_len"] = np.fromiter((len(s) for s in xa), np.int32, len(xa))
    f["cand_name_len"] = np.fromiter((len(s) for s in g(B, "name_b", b)), np.int32, len(b))
    f["s1_name_len"] = np.fromiter((len(s) for s in g(A, "name_b", a)), np.int32, len(a))
    f["cand_nonascii"] = g(B, "nonascii", b)
    f["is_s3"] = np.int8(is_s3)
    # context: how this candidate compares with the other candidates of the same Source 1 record
    v = f.nsk_tset.to_numpy() + f.ask_tset.to_numpy()
    grp = pd.DataFrame({"a": a, "v": v}).groupby("a")["v"]
    f["rank_s1"] = grp.rank(ascending=False, method="min").to_numpy(np.float32)
    f["gap_s1"] = (grp.transform("max").to_numpy() - v).astype(np.float32)
    f["n_cand_s1"] = grp.transform("size").to_numpy(np.int32)
    return f

# ---------------------------------------------------------------------
# 5. Scoring (same formula as the leaderboard) and the final decision rule
# ---------------------------------------------------------------------
def macro_f05(pred, truth, ids):
    sc = []
    for s in ids:
        p, t = pred.get(s, set()), truth.get(s, set())
        if not t:
            sc.append(1.0 if not p else 0.0); continue
        tp = len(p & t)
        if tp == 0:
            sc.append(0.0); continue
        P, R = tp / len(p), tp / len(t)
        sc.append(1.25 * P * R / (0.25 * P + R))
    return float(np.mean(sc))

def decide(pairs, thr):
    """pairs: DataFrame(s1_id, cand_id, p). Keep p >= thr; each S2/S3 record goes to at most one S1 (its best)."""
    d = pairs[pairs.p >= thr].sort_values("p", ascending=False).drop_duplicates("cand_id")
    return d.groupby("s1_id")["cand_id"].apply(set).to_dict()

def write_tsv(path, s1_ids, mapping, col):
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(f"source1_entity_id\t{col}\n")
        for s in s1_ids:
            fh.write(s + "\t" + ",".join(sorted(mapping.get(s, ()))) + "\n")


In [ ]:
ROOT = os.environ.get("ROOT", "/kaggle/input")      # the notebook searches here for the .tsv files
WORK = os.environ.get("WORK", "/kaggle/working")    # models + outputs are saved here
N_TRAIN_S1 = int(os.environ.get("N_TRAIN_S1", 200_000))  # training businesses (v1 used 150k)
BATCH = int(os.environ.get("BATCH", 100_000))       # test businesses processed at once (lower it if memory runs out)
KEEP_P = 0.01        # stage-1 filter: candidates below this chance are dropped before stage 2
K_CTX = 8            # stage 2 compares each candidate with this many of the business's best candidates
SMOKE = 0             # 1 = tiny quick test run (for checking the code only)
os.makedirs(WORK, exist_ok=True)
log("CPU cores:", N_CPU, "| training businesses:", N_TRAIN_S1, "| smoke test:", bool(SMOKE))


In [ ]:
# =====================================================================
# Version 2: a second model that looks at ALL candidates of a business together.
# One real business shows up about 3 times across Source 2 and Source 3, and those copies look alike.
# Stage 1 (like v1) scores each candidate on its own. Stage 2 then asks: "does this candidate look like the
# other strong candidates of the same business?" and decides per business instead of one fixed cut-off.
# =====================================================================
from sklearn.model_selection import GroupKFold

def load_src(name):
    """Load + clean one Source 2/3 file (in a smoke test, only a random 1M-row slice).
    The cleaned table is saved to disk the first time, so later calls just read it back.
    (Cleaning uses several worker processes; doing it early, while memory is still empty, avoids out-of-memory.)"""
    path = f"{WORK}/prep_{name}.parquet"
    if os.path.exists(path):
        return pd.read_parquet(path)
    B = load(ROOT, name)
    if SMOKE:
        B = B.sample(min(len(B), 1_000_000), random_state=0).reset_index(drop=True)
    B = prepare(B); B.to_parquet(path)
    return B

def shortlist_with_features(Ac, idx, Bc, is_s3):
    """Shortlist + stage-1 features for Source 1 rows Ac against one pool Bc."""
    P = idx.query(Ac)
    if len(P) == 0:
        return None, None
    F = features(P, Ac, Bc, is_s3)
    b = P.b.to_numpy()
    M = pd.DataFrame({"a": P.a.to_numpy(), "cand_id": Bc.entity_id.to_numpy()[b],
                      "cname": Bc.name_sk.to_numpy()[b], "caddr": Bc.addr_sk.to_numpy()[b]})
    return M, F

S2_EXTRA = ["p1", "p1_rank", "p1_max", "p1_2nd", "p1_sum", "p1_rel", "n_hi", "n_kept",
            "sup_max", "sup_sum", "sup_cnt", "sup_name_max", "sup_addr_max", "sup_other_src"]

def stage2_features(key, p1, cname, caddr, is_s3, K=K_CTX):
    """Context features. key: business of each row; p1: stage-1 chance; cname/caddr: candidate's cleaned text."""
    d = pd.DataFrame({"key": key, "p1": p1.astype(np.float32), "row": np.arange(len(key)), "s3": is_s3})
    d = d.sort_values(["key", "p1"], ascending=[True, False], kind="stable")
    g = d.groupby("key", sort=False)
    d["p1_rank"] = (g.cumcount() + 1).astype(np.float32)
    d["p1_max"] = g.p1.transform("max")
    d["p1_2nd"] = d.key.map(d[d.p1_rank == 2].set_index("key").p1).fillna(0).to_numpy(np.float32)
    d["p1_sum"] = g.p1.transform("sum")
    d["p1_rel"] = d.p1 / d.p1_max.clip(lower=1e-6)
    d["n_hi"] = (d.p1 > 0.5).groupby(d.key).transform("sum").astype(np.float32)
    d["n_kept"] = g.p1.transform("size").astype(np.float32)
    top = d[d.p1_rank <= K][["key", "row", "p1", "s3"]]
    j = top.merge(top, on="key", suffixes=("", "_o"))
    j = j[j.row != j.row_o]
    r, ro = j.row.to_numpy(), j.row_o.to_numpy()
    sn = cpdist(cname[r].tolist(), cname[ro].tolist(), scorer=fuzz.token_set_ratio, workers=N_CPU).astype(np.float32) / 100
    sa = cpdist(caddr[r].tolist(), caddr[ro].tolist(), scorer=fuzz.token_set_ratio, workers=N_CPU).astype(np.float32) / 100
    po = j.p1_o.to_numpy()
    s = (sn + sa) / 2
    j = pd.DataFrame({"row": r, "w": po * s, "strong": ((s > 0.85) & (po > 0.5)).astype(np.int8),
                      "wn": po * sn, "wa": po * sa, "wx": np.where(j.s3.to_numpy() != j.s3_o.to_numpy(), po * s, 0)})
    agg = j.groupby("row").agg(sup_max=("w", "max"), sup_sum=("w", "sum"), sup_cnt=("strong", "sum"),
                               sup_name_max=("wn", "max"), sup_addr_max=("wa", "max"), sup_other_src=("wx", "max"))
    d = d.set_index("row").sort_index().join(agg)
    return d[S2_EXTRA].fillna(0).astype(np.float32).reset_index(drop=True)

def stage2_in_chunks(key, *cols, chunk=200_000):
    """stage2_features for many businesses, a block of businesses at a time to save memory."""
    order = np.argsort(key, kind="stable"); ks = key[order]
    uk = np.unique(ks); out = []
    for i in range(0, len(uk), chunk):
        lo, hi = np.searchsorted(ks, uk[i]), np.searchsorted(ks, uk[min(i + chunk, len(uk)) - 1], side="right")
        sel = order[lo:hi]
        f = stage2_features(key[sel], *[c[sel] for c in cols]); f.index = sel; out.append(f)
    return pd.concat(out).sort_index()

def decide2(pairs, t_abs, r_rel):
    """Keep a candidate if p >= t_abs AND p >= r_rel x (best p of that business).
    Each Source 2/3 record goes to at most one business (the one it fits best)."""
    d = pairs[pairs.p >= t_abs]
    d = d[d.p >= r_rel * d.groupby("s1_id").p.transform("max")]
    d = d.sort_values("p", ascending=False).drop_duplicates("cand_id")
    return d.groupby("s1_id")["cand_id"].apply(set).to_dict()


In [ ]:
# ---------- 1. shortlist + stage-1 features for a sample of training businesses ----------
for src in ("train_source2", "train_source3"):          # clean both sources first (saved to disk)
    log("cleaning", src); B = load_src(src); log("cleaned", src, B.shape); del B; gc.collect()
log("loading train Source 1 + ground truth")
s1 = load(ROOT, "train_source1"); gt = load(ROOT, "train_ground_truth")
s1 = s1.sample(N_TRAIN_S1 if not SMOKE else 20_000, random_state=0).reset_index(drop=True)
truth = {r.source1_entity_id: set(x for x in r.matched_entity_ids.split(",") if x) for r in gt.itertuples()}
del gt; gc.collect()
A = prepare(s1); log("prepared S1 sample", A.shape)
Ms, Fs = [], []
for src, is_s3 in (("train_source2", 0), ("train_source3", 1)):
    log("loading", src); B = load_src(src); log("prepared", src, B.shape)
    for country in A.country.unique():
        Ac = A[A.country == country].reset_index(drop=True); Bc = B[B.country == country].reset_index(drop=True)
        if len(Ac) == 0 or len(Bc) == 0: continue
        M, F = shortlist_with_features(Ac, Index(Bc), Bc, is_s3)
        if M is None: continue
        M["s1_id"] = Ac.entity_id.to_numpy()[M.a.to_numpy()]; M["s3"] = np.int8(is_s3)
        Ms.append(M.drop(columns="a")); Fs.append(F); log(src, country, "pairs", len(M)); gc.collect()
    del B; gc.collect()
X = pd.concat(Fs, ignore_index=True); M = pd.concat(Ms, ignore_index=True); del Ms, Fs; gc.collect()
FEAT1 = list(X.columns)
y = np.fromiter((c in truth.get(s, ()) for s, c in zip(M.s1_id, M.cand_id)), bool, len(M)).astype(np.int8)
all_true = sum(len(truth[s]) for s in A.entity_id)
log("pairs", len(M), "per S1", round(len(M) / len(A), 1), "| shortlist recall", round(y.sum() / all_true, 4))

# ---------- 2. stage 1: out-of-fold chances (each business scored by a model that never saw it) ----------
P1 = dict(n_estimators=400, learning_rate=0.08, num_leaves=255, min_child_samples=100, subsample=0.8,
          subsample_freq=1, colsample_bytree=0.8, n_jobs=N_CPU, verbose=-1)
if SMOKE: P1["n_estimators"] = 60
p1 = np.zeros(len(M), np.float32)
for k, (tr, te) in enumerate(GroupKFold(n_splits=3).split(X, y, groups=M.s1_id)):
    m1 = lgb.LGBMClassifier(**P1).fit(X.iloc[tr], y[tr]); p1[te] = m1.predict_proba(X.iloc[te])[:, 1]
    log("stage-1 fold", k + 1, "of 3 done"); del m1; gc.collect()
m1 = lgb.LGBMClassifier(**P1).fit(X, y); m1.booster_.save_model(f"{WORK}/stage1.txt"); del m1; gc.collect()
log("stage-1 full model saved")

# ---------- 3. keep only candidates with a real chance, then build stage-2 features ----------
keep = p1 >= KEEP_P
log("kept after stage 1:", int(keep.sum()), "pairs,", round(keep.sum() / len(A), 1), "per S1 | true matches kept",
    round(y[keep].sum() / all_true, 4))
X = X[keep].reset_index(drop=True); M = M[keep].reset_index(drop=True); y = y[keep]; p1 = p1[keep]; gc.collect()
S2F = stage2_in_chunks(M.s1_id.to_numpy(), p1, M.cname.to_numpy(), M.caddr.to_numpy(), M.s3.to_numpy())
X2 = pd.concat([X, S2F.reset_index(drop=True)], axis=1); del X, S2F; gc.collect()
FEAT2 = list(X2.columns); log("stage-2 features", X2.shape)

# ---------- 4. stage 2 model, checked on 20% held-out businesses ----------
ids = A.entity_id.to_numpy()
val_ids = set(np.random.default_rng(42).choice(ids, int(0.2 * len(ids)), replace=False))
iv = M.s1_id.isin(val_ids).to_numpy(); vl = sorted(val_ids); vt = {s: truth[s] for s in vl}
log("baseline all-empty", round(macro_f05({}, vt, vl), 4))
log("ceiling (perfect model on this shortlist)",
    round(macro_f05(M[iv & (y == 1)].groupby("s1_id").cand_id.apply(set).to_dict(), vt, vl), 4))
v1p = M[iv][["s1_id", "cand_id"]].assign(p=p1[iv])
log("stage-1 only (like v1), best val F0.5",
    round(max(macro_f05(decide(v1p, t), vt, vl) for t in np.arange(0.4, 0.86, 0.05)), 4))
P2 = dict(n_estimators=4000, learning_rate=0.05, num_leaves=255, min_child_samples=100, subsample=0.8,
          subsample_freq=1, colsample_bytree=0.8, n_jobs=N_CPU, verbose=-1)
m2 = lgb.LGBMClassifier(**P2).fit(X2[~iv], y[~iv], eval_set=[(X2[iv], y[iv])],
                                  callbacks=[lgb.early_stopping(100, verbose=False)])
n_trees = max(m2.best_iteration_ or P2["n_estimators"], 50); log("stage-2 trees", n_trees)
v2p = M[iv][["s1_id", "cand_id"]].assign(p=m2.predict_proba(X2[iv])[:, 1]); del m2; gc.collect()
best = (0, None)
for t in np.arange(0.3, 0.91, 0.05):
    for r in (0.0, 0.2, 0.3, 0.4, 0.5, 0.6):
        s = macro_f05(decide2(v2p, t, r), vt, vl)
        if s > best[0]: best = (s, (round(float(t), 2), r))
log("STAGE-2 BEST val F0.5", round(best[0], 4), "at cut-off / relative-to-best", best[1], "(v1 was 0.9529)")
final = lgb.LGBMClassifier(**{**P2, "n_estimators": n_trees}).fit(X2, y)
final.booster_.save_model(f"{WORK}/stage2.txt")
json.dump({"t_abs": best[1][0], "r_rel": best[1][1], "val_f05": best[0], "features1": FEAT1, "features2": FEAT2},
          open(f"{WORK}/model_v2_meta.json", "w"))
log("saved. top stage-2 features:\n" + pd.Series(final.feature_importances_, FEAT2).sort_values(ascending=False).head(12).to_string())
del X2, M, y, p1, final, A, s1; gc.collect()
for src in ("train_source2", "train_source3"):
    os.remove(f"{WORK}/prep_{src}.parquet")              # free disk space


In [ ]:
# ---------- stage 1 on every test business, one batch at a time, saved to disk (safe to resume) ----------
PARTS = f"{WORK}/parts_v2"; os.makedirs(PARTS, exist_ok=True)
meta = json.load(open(f"{WORK}/model_v2_meta.json"))
b1 = lgb.Booster(model_file=f"{WORK}/stage1.txt"); b2 = lgb.Booster(model_file=f"{WORK}/stage2.txt")
for src in ("test_source2", "test_source3"):            # clean both sources first (saved to disk)
    log("cleaning", src); B = load_src(src); log("cleaned", src, B.shape); del B; gc.collect()
log("loading test Source 1"); T = load(ROOT, "test_source1")
if SMOKE: T = T.groupby("country").head(10_000).reset_index(drop=True)
A = prepare(T); del T; log("prepared", A.shape); print(A.country.value_counts().to_string())
s1_ids = A.entity_id.to_numpy()
for src, is_s3 in (("test_source2", 0), ("test_source3", 1)):
    todo = [(c, i) for c in A.country.unique() for i in range(0, int((A.country == c).sum()), BATCH)
            if not os.path.exists(f"{PARTS}/{src}_{c}_{i}.parquet")]
    if not todo:
        log(src, "already done"); continue
    log("loading", src); B = load_src(src); log("prepared", src, B.shape)
    for country in A.country.unique():                 # any country label works, France included
        a_rows = np.flatnonzero(A.country.to_numpy() == country)
        Bc = B[B.country == country].reset_index(drop=True)
        if len(Bc) == 0 or not any(c == country for c, _ in todo): continue
        idx = Index(Bc)
        for i in range(0, len(a_rows), BATCH):
            out = f"{PARTS}/{src}_{country}_{i}.parquet"
            if os.path.exists(out): continue
            rows = a_rows[i:i + BATCH]; Ac = A.iloc[rows].reset_index(drop=True)
            M, F = shortlist_with_features(Ac, idx, Bc, is_s3)
            if M is None:
                pd.DataFrame({"s1_row": np.zeros(0, np.int32)}).to_parquet(out); continue
            p1 = b1.predict(F[meta["features1"]].to_numpy(np.float32), num_threads=N_CPU).astype(np.float32)
            k = p1 >= KEEP_P
            part = F[k].reset_index(drop=True).astype(np.float32)
            part["s1_row"] = rows[M.a.to_numpy()[k]].astype(np.int32); part["cand_id"] = M.cand_id.to_numpy()[k]
            part["cname"] = M.cname.to_numpy()[k]; part["caddr"] = M.caddr.to_numpy()[k]
            part["s3"] = np.int8(is_s3); part["p1"] = p1[k]
            part.to_parquet(out)
            log(src, country, f"batch {i // BATCH + 1}/{-(-len(a_rows) // BATCH)}", "shortlist", len(M), "kept", int(k.sum()))
            del M, F, part, p1; gc.collect()
        del idx, Bc; gc.collect()
    del B; gc.collect()

# ---------- stage 2: look at each business's candidates from both sources together, then decide ----------
log("stage 2")
allp = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(f"{PARTS}/*.parquet"))], ignore_index=True)
allp = allp[allp.s1_row.notna()].reset_index(drop=True)
S2F = stage2_in_chunks(allp.s1_row.to_numpy(), allp.p1.to_numpy(), allp.cname.to_numpy(), allp.caddr.to_numpy(),
                       allp.s3.to_numpy())
X2 = pd.concat([allp[meta["features1"]], S2F.reset_index(drop=True)], axis=1)[meta["features2"]]
p = b2.predict(X2.to_numpy(np.float32), num_threads=N_CPU); del X2, S2F; gc.collect()
pairs = pd.DataFrame({"s1_id": s1_ids[allp.s1_row.to_numpy().astype(int)], "cand_id": allp.cand_id.to_numpy(), "p": p})
match = decide2(pairs, meta["t_abs"], meta["r_rel"])
cmap = pairs.groupby("s1_id").cand_id.agg(list).to_dict(); del allp, pairs; gc.collect()
os.makedirs(f"{WORK}/output", exist_ok=True)
write_tsv(f"{WORK}/output/matching_results.tsv", s1_ids, match, "matched_entity_ids")
write_tsv(f"{WORK}/output/candidate_pairs.tsv", s1_ids, cmap, "candidate_entity_ids")
n = sum(1 for s in s1_ids if match.get(s))
log("done:", len(s1_ids), "rows,", n, "with matches,", len(s1_ids) - n, "empty; avg matches",
    round(sum(len(v) for v in match.values()) / len(s1_ids), 2))
for src in ("test_source2", "test_source3"):
    os.remove(f"{WORK}/prep_{src}.parquet")              # free disk space (cleaned copies no longer needed)


In [ ]:
validator = glob.glob(f'{ROOT}/**/validate_submission.py', recursive=True)[0]
test_dir = os.path.dirname(find_file(ROOT, 'test_source1')[0])
!python3 {validator} --matching {WORK}/output/matching_results.tsv --candidate {WORK}/output/candidate_pairs.tsv --test-dir {test_dir}

**Done.** Open this version's **Output** tab, go into `output/`, and download **`matching_results.tsv`**. Keep `candidate_pairs.tsv` for the final zip. You can delete the `parts_v2/` folder.